# Tutorial 7: Compare Feature Extraction vs. Fine-Tuning step by step  

### Cell 1: Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### Cell 2: Load and Preprocess CIFAR-10

In [14]:
#transform = transforms.Compose([
#    transforms.Resize((224, 224)), 
#    transforms.ToTensor(),
#    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
#])

#trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
#trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)

#testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
#testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

# Updated Cell 2: Optimized for Speed
transform = transforms.Compose([
    # We remove Resize(224) to save hours of training time
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# Use a smaller subset of data if you just want to see the code work instantly
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
# Increase batch_size to 128 to process images faster
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

print(f"Batches per epoch reduced from 1563 to {len(trainloader)}")

Batches per epoch reduced from 1563 to 391


### Cell 3: Task - Feature Extraction with ResNet50

In [15]:
# Load Pretrained ResNet50
model_vgg_ft = models.vgg16(weights='IMAGENET1K_V1')

# Freeze all layers initially
for param in model_vgg_ft.parameters():
    param.requires_grad = False

# Task: Unfreeze the last 5 layers 
for param in list(model_vgg_ft.features.parameters())[-5:]:
    param.requires_grad = True

# Match the classifier to CIFAR-10 classes 
model_vgg_ft.classifier[6] = nn.Linear(4096, 10)
model_vgg_ft = model_vgg_ft.to(device)

print("ResNet50 Feature Extraction model ready.")

ResNet50 Feature Extraction model ready.


### Cell 4: Task - Fine-Tuning with VGG16

In [16]:
# Load Pretrained VGG16
model_vgg_ft = models.vgg16(weights='IMAGENET1K_V1')

# Freeze all layers initially 
for param in model_vgg_ft.parameters():
    param.requires_grad = False

# Task: Unfreeze the last 5 layers 
for param in list(model_vgg_ft.features.parameters())[-5:]:
    param.requires_grad = True

# Match the classifier to CIFAR-10 classes 
model_vgg_ft.classifier[6] = nn.Linear(4096, 10)
model_vgg_ft = model_vgg_ft.to(device)

print("VGG16 Fine-Tuning model ready.")

VGG16 Fine-Tuning model ready.


### Cell 5: Training and Evaluation Function

In [18]:
def train_and_evaluate(model, name, is_fine_tuning=False):
    criterion = nn.CrossEntropyLoss()
    
    lr = 1e-5 if is_fine_tuning else 1e-3
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    print(f"--- Starting: {name} ---")
    model.train()
    
    # Tutorial Task: Number of Epochs 
    epochs = 2 
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            if i % 100 == 99:    # Print every 100 mini-batches
                print(f'[{name}] Epoch {epoch + 1}, Batch {i + 1}: loss {running_loss / 100:.3f}')
                running_loss = 0.0
            
    # Evaluation Step 
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"RESULT: Accuracy of {name} is {accuracy:.2f}%\n")
    return accuracy

### Cell 6: Run Comparison Task

In [19]:
acc_resnet = train_and_evaluate(model_resnet_fe, "ResNet50 (Feature Extraction)", is_fine_tuning=False)
acc_vgg = train_and_evaluate(model_vgg_ft, "VGG16 (Fine-Tuning)", is_fine_tuning=True)

print(f"\nFinal Results Comparison:")
print(f"ResNet50 FE: {acc_resnet}%")
print(f"VGG16 FT: {acc_vgg}%")

--- Starting: ResNet50 (Feature Extraction) ---
[ResNet50 (Feature Extraction)] Epoch 1, Batch 100: loss 1.848
[ResNet50 (Feature Extraction)] Epoch 1, Batch 200: loss 1.611
[ResNet50 (Feature Extraction)] Epoch 1, Batch 300: loss 1.563
[ResNet50 (Feature Extraction)] Epoch 2, Batch 100: loss 1.453
[ResNet50 (Feature Extraction)] Epoch 2, Batch 200: loss 1.461
[ResNet50 (Feature Extraction)] Epoch 2, Batch 300: loss 1.490
RESULT: Accuracy of ResNet50 (Feature Extraction) is 50.92%

--- Starting: VGG16 (Fine-Tuning) ---
[VGG16 (Fine-Tuning)] Epoch 1, Batch 100: loss 2.401
[VGG16 (Fine-Tuning)] Epoch 1, Batch 200: loss 2.052
[VGG16 (Fine-Tuning)] Epoch 1, Batch 300: loss 1.738
[VGG16 (Fine-Tuning)] Epoch 2, Batch 100: loss 1.392
[VGG16 (Fine-Tuning)] Epoch 2, Batch 200: loss 1.320
[VGG16 (Fine-Tuning)] Epoch 2, Batch 300: loss 1.243
RESULT: Accuracy of VGG16 (Fine-Tuning) is 63.42%


Final Results Comparison:
ResNet50 FE: 50.92%
VGG16 FT: 63.42%
